# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Lab 4: Build and Use a Mini-PKI

**60 minutes guided · 90–120 minutes independently.** Notebook (see course website) · Instructor (see course website) · Solutions (see course website)

## Goal and preparation

Create a root, intermediate, server certificate and client certificate. Inspect SAN, validity, Basic Constraints and EKU. Configure trust for a real TLS handshake and prove that wrong identities and missing credentials fail. Complete Sessions 6–7 and the setup (see course website). Helpers are embedded below in the notebook; inspect their source before using them.



```mermaid
flowchart LR
    R["Root trusted locally"] --> I["Intermediate issues leaves"]
    I --> S["Server: invoice.test"]
    I --> C["Client: client-auth purpose"]
    S --> T["TLS handshake in memory"]
    C --> T
    T --> A["Application authorization is still separate"]
```

The diagram separates issuing authority, handshake authentication and permissions. No system trust store is modified.

## Task one: create and inspect the hierarchy — 15 minutes

Open `make_pki` in the helper cell. Identify each `.sign` issuer key and the CA/path-length constraints. Predict which private key signs the server leaf and why the server should send the intermediate rather than expect it to become a trust anchor.


In [ ]:
keys, certs = make_pki()
for name, cert in certs.items():
    print(name, cert.subject.rfc4514_string(), cert.issuer.rfc4514_string())
    print('  CA:', cert.extensions.get_extension_for_class(x509.BasicConstraints).value.ca)
    print('  Valid until:', cert.not_valid_after_utc.isoformat())
assert certs['server'].issuer == certs['intermediate'].subject
print('PASS: generated and inspected four teaching certificates')


Record root and intermediate roles, leaf SAN, and client/server EKU. A subject-name match in this inspection is not chain validation; the next experiment uses the real verifier.

## Task two: predict success and failures — 15 minutes


In [ ]:
def observe_tls(case='valid'):
    settings = {
        'valid': {}, 'wrong SAN': {'hostname': 'other.test'},
        'unknown root': {'trust_root': False}, 'expired': {'expired': True},
        'missing intermediate': {'include_intermediate': False},
        'mTLS valid': {'mtls': True},
        'mTLS no client': {'mtls': True, 'send_client': False},
        'mTLS wrong purpose': {'mtls': True, 'client_wrong_eku': True},
    }
    try:
        result = tls_trial(**settings[case])
    except ssl.SSLError as error:
        return 'REJECTED: ' + str(error)
    return 'ACCEPTED: ' + str(result)

print(observe_tls('valid'))
for case in ('wrong SAN', 'unknown root', 'expired', 'missing intermediate',
             'mTLS no client', 'mTLS wrong purpose'):
    assert observe_tls(case).startswith('REJECTED:')
assert observe_tls('mTLS valid').startswith('ACCEPTED:')
print('PASS: six negative TLS cases rejected and mTLS succeeded')


Optional notebook control (the direct `observe_tls('wrong SAN')` call is equivalent):


In [ ]:
if 'get_ipython' in globals():
    import ipywidgets as widgets
    from IPython.display import display
    display(widgets.interactive(observe_tls, case=['valid', 'wrong SAN', 'unknown root',
        'expired', 'missing intermediate', 'mTLS valid', 'mTLS no client', 'mTLS wrong purpose']))


Write which requirement rejected each case. Do not “repair” the experiment by turning verification off.

## Task three: implement a verified connection wrapper — 20 minutes

Implement `learner_connect(hostname, require_client, client_present)` by calling `tls_trial` with matching arguments. Keep trust and hostname checks enabled. Return its result; let required failures raise `ssl.SSLError`. This is a wrapper exercise, not a new certificate validator.


In [ ]:
def learner_connect(hostname, require_client, client_present):
    raise NotImplementedError('Implement verified TLS/mTLS configuration')

def check_connection(candidate):
    result = candidate('invoice.test', False, False)
    assert result['version'] == 'TLSv1.3' and not result['client_authenticated']
    assert candidate('invoice.test', True, True)['client_authenticated']
    expect_rejection(lambda: candidate('wrong.test', False, False), ssl.SSLError)
    expect_rejection(lambda: candidate('invoice.test', True, False), ssl.SSLError)

try:
    check_connection(learner_connect)
except NotImplementedError:
    print('NOT ATTEMPTED: learner TLS wrapper')
else:
    print('PASS: learner TLS wrapper')


<details><summary>Hints</summary><p>The relevant arguments are hostname, mtls and send_client. Do not pass trust_root=False for a normal connection. Client authentication is required by server policy, not by whether the client happens to offer a certificate.</p></details>

## Reference and debrief — 10 minutes

Read after attempting the task. Reference success is not learner completion.


In [ ]:
def reference_connect(hostname, require_client, client_present):
    return tls_trial(hostname=hostname, mtls=require_client, send_client=client_present)
check_connection(reference_connect)
print('PASS: supplied Lab 4 reference checks')


Submit your wrapper, the failure table and a paragraph describing what the lab does **not** establish: online revocation, authorization, production key custody and PQ TLS negotiation. Explain why installing an arbitrary root is not a safe fix for an unknown issuer.

Extension: issue a client certificate for a second workload and sketch a mapping from its authenticated identity to allowed invoice operations. Keep that authorization decision distinct from certificate validity. See the worked solution (see course website).


In [ ]:
print("PASS: completed lab-04-mini-pki demonstrations; learner status is reported separately")
